# Preference Diversity, Mobility, and Movement Behaviour in an Extended Schelling Model

This notebook answers the research question: **How do preference diversity, mobility, and movement behaviour interact to affect final segregation and stabilisation time?**

It is designed to run from top to bottom from a clean checkout. If result CSV files have not yet been generated, the notebook prints the required commands and skips result-dependent figures rather than failing.

## 1. Project background and research question

The classic Schelling model demonstrates how local residential preferences can produce global segregation. This project extends the model to test whether heterogeneity in preferences interacts with agents' movement radius and relocation strategy. The primary outcomes are the final segregation index and the number of iterations before stopping.

## 2. Novel extensions to the classic model

The implementation adds three experimental extensions: heterogeneous similarity preferences, limited mobility, and three destination-selection behaviours (`random`, `improving`, and `best_fit`). An optional social-influence rule moves each preference toward the mean preference of occupied neighbours.

## 3. Model assumptions and update rules

- The environment is a finite rectangular grid with two groups and vacant cells.
- Similarity is the proportion of occupied Moore neighbours belonging to the same group.
- An agent is satisfied when similarity is greater than or equal to its preference.
- Mobility uses Chebyshev distance and restricts accessible vacant cells.
- Dissatisfied agents move according to their configured behaviour.
- When enabled, social influence uses a synchronous pre-update preference snapshot.
- A run stops when neither movement nor preference updates occur, or at the maximum iteration count.

## 4. Experimental factors and hypotheses

Preference diversity keeps the population mean fixed at 0.5: homogeneous agents use 0.5, low diversity uses symmetric samples within [0.4, 0.6], and high diversity uses symmetric samples within [0.2, 0.8]. Mobility levels are 1, 3, and 5. All three behaviours are crossed with these factors.

Hypotheses:
1. Preference diversity changes final segregation and stabilisation time.
2. The effect of mobility depends on movement behaviour.
3. `best_fit` produces greater segregation than `random`.
4. Social influence reduces preference variation and changes stabilisation.
5. Interactions are important and cannot be inferred from main effects alone.

## 5. Reproducibility and seed strategy

The pilot uses seeds 0-9 and the final design uses seeds 0-29 for every condition. Reusing the same seeds creates paired stochastic comparisons. The runner records all parameters and metrics in CSV files and refuses to overwrite an existing file unless explicitly requested.

In [ ]:
from pathlib import Path
from collections import defaultdict
import csv
import math
import statistics

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data'
print(f'Project root: {ROOT}')
print(f'Data directory: {DATA_DIR}')

Generate data from the project root with one of the following commands:

```bash
python -B -m utils.experiment_runner --design smoke
python -B -m utils.experiment_runner --design pilot
python -B -m utils.experiment_runner --design main
python -B -m utils.experiment_runner --design social
```

The smoke design is only a workflow check and must not be used for substantive conclusions.

## 6. Data loading and validation

In [ ]:
INTEGER_FIELDS = {'seed', 'width', 'height', 'mobility', 'max_iterations', 'iterations',
                  'total_moves', 'total_preference_updates', 'iteration', 'moves',
                  'preference_updates'}
NULLABLE_INTEGER_FIELDS = {'stabilisation_time'}
FLOAT_FIELDS = {'vacancy_rate', 'group_split', 'preference_lower', 'preference_upper',
                'initial_preference_mean', 'initial_preference_std', 'influence_strength',
                'final_segregation_index', 'final_satisfaction_rate',
                'final_preference_mean', 'final_preference_std',
                'satisfaction_rate', 'segregation_index'}
BOOLEAN_FIELDS = {'social_influence', 'stabilised'}

def load_csv(path):
    path = Path(path)
    if not path.exists():
        return []
    with path.open(encoding='utf-8', newline='') as source:
        rows = list(csv.DictReader(source))
    for row in rows:
        for key, value in list(row.items()):
            if key in INTEGER_FIELDS:
                row[key] = int(value)
            elif key in NULLABLE_INTEGER_FIELDS:
                row[key] = int(value) if value not in ('', 'None', 'nan') else None
            elif key in FLOAT_FIELDS:
                row[key] = float(value)
            elif key in BOOLEAN_FIELDS:
                row[key] = value.lower() == 'true'
    return rows

def first_available(names):
    for name in names:
        path = DATA_DIR / name
        if path.exists():
            return path
    return None

main_path = first_available(['main_experiment_results.csv', 'pilot_results.csv', 'smoke_results.csv'])
social_path = first_available(['social_influence_results.csv'])
history_path = first_available(['main_experiment_history.csv', 'pilot_history.csv',
                                'social_influence_history.csv', 'smoke_history.csv'])

main_rows = load_csv(main_path) if main_path else []
social_rows = load_csv(social_path) if social_path else []
history_rows = load_csv(history_path) if history_path else []
print(f'Main/pilot rows: {len(main_rows)} from {main_path}')
print(f'Social comparison rows: {len(social_rows)} from {social_path}')
print(f'Iteration-history rows: {len(history_rows)} from {history_path}')

In [ ]:
REQUIRED_COLUMNS = {
    'run_id', 'seed', 'preference_diversity', 'mobility', 'behaviour',
    'social_influence', 'final_segregation_index', 'iterations',
    'stabilised', 'final_satisfaction_rate', 'total_preference_updates'
}

def validate_summary(rows, label):
    if not rows:
        print(f'{label}: no data available; result-dependent analysis will be skipped.')
        return
    missing = REQUIRED_COLUMNS - set(rows[0])
    assert not missing, f'{label} is missing columns: {sorted(missing)}'
    run_ids = [row['run_id'] for row in rows]
    assert len(run_ids) == len(set(run_ids)), f'{label} contains duplicate run IDs'
    assert all(0 <= row['final_segregation_index'] <= 1 for row in rows)
    assert all(0 <= row['final_satisfaction_rate'] <= 1 for row in rows)
    print(f'{label}: validation passed for {len(rows)} runs.')

validate_summary(main_rows, 'Main/pilot results')
validate_summary(social_rows, 'Social-influence results')

In [ ]:
def grouped_stats(rows, factor, metric, order=None, filters=None):
    filters = filters or {}
    groups = defaultdict(list)
    for row in rows:
        if all(row.get(key) == value for key, value in filters.items()):
            groups[row[factor]].append(row[metric])
    keys = [key for key in (order or groups.keys()) if key in groups]
    means = [statistics.fmean(groups[key]) for key in keys]
    errors = [
        1.96 * statistics.stdev(groups[key]) / math.sqrt(len(groups[key]))
        if len(groups[key]) > 1 else 0.0
        for key in keys
    ]
    return keys, means, errors

def bar_plot(rows, factor, metric, order, title, xlabel, ylabel, filters=None):
    keys, means, errors = grouped_stats(rows, factor, metric, order, filters)
    if not keys:
        print(f'Skipped: {title} (no matching data).')
        return
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar([str(key) for key in keys], means, yerr=errors, capsize=4, color='#4C78A8')
    ax.set(title=title, xlabel=xlabel, ylabel=ylabel)
    ax.grid(axis='y', alpha=0.25)
    plt.tight_layout()
    plt.show()

## 7. Results and visualisations

Error bars below are approximate 95% confidence intervals across repeated seeds. Smoke-test results are workflow evidence only.

In [ ]:
bar_plot(
    main_rows, 'preference_diversity', 'final_segregation_index',
    ['homogeneous', 'low', 'high'],
    'Final segregation by preference diversity',
    'Preference diversity', 'Mean final segregation index'
)

**Figure caption.** Mean final segregation for each preference-diversity condition, averaged across mobility, behaviour, and repeated seeds. Interpret the direction and uncertainty only after the final experiment has been generated.

In [ ]:
stabilised_rows = [row for row in main_rows if row.get('stabilised')]
non_stabilised = len(main_rows) - len(stabilised_rows)
if non_stabilised:
    print(f'Note: {non_stabilised} of {len(main_rows)} runs did not stabilise '
          f'and are excluded from stopping-time analysis.')

bar_plot(
    stabilised_rows, 'mobility', 'stabilisation_time', [1, 3, 5],
    'Stabilisation time by mobility (stabilised runs only)', 'Mobility radius',
    'Mean iterations to stabilise'
)

**Figure caption.** Mean stopping time by movement radius. Runs that reach the maximum iteration count remain part of this descriptive comparison and should also be discussed using the stabilised indicator.

In [ ]:
bar_plot(
    main_rows, 'behaviour', 'final_segregation_index',
    ['random', 'improving', 'best_fit'],
    'Final segregation by movement behaviour',
    'Movement behaviour', 'Mean final segregation index'
)

**Figure caption.** Comparison of relocation strategies. Differences should be interpreted together with the interaction plots because averaging can hide condition-specific effects.

In [ ]:
def interaction_plot(rows):
    if not rows:
        print('Skipped: interaction plot (no data).')
        return
    diversities = ['homogeneous', 'low', 'high']
    behaviours = ['random', 'improving', 'best_fit']
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for ax, diversity in zip(axes, diversities):
        for behaviour in behaviours:
            x, y, _ = grouped_stats(
                rows, 'mobility', 'final_segregation_index', [1, 3, 5],
                {'preference_diversity': diversity, 'behaviour': behaviour}
            )
            if x:
                ax.plot(x, y, marker='o', label=behaviour)
        ax.set_title(diversity.capitalize())
        ax.set_xlabel('Mobility radius')
        ax.grid(alpha=0.25)
    axes[0].set_ylabel('Mean final segregation index')
    axes[-1].legend(title='Behaviour')
    fig.suptitle('Interaction between preference diversity, mobility, and behaviour')
    plt.tight_layout()
    plt.show()

interaction_plot(main_rows)

**Figure caption.** Mobility effects are displayed separately for each preference-diversity condition and movement behaviour. Non-parallel lines provide descriptive evidence of interaction; formal causal claims require appropriate statistical analysis.

In [ ]:
def paired_bar_plot(rows, metric, title, ylabel):
    if not rows:
        print(f'Skipped: {title} (no data).')
        return
    by_seed = {}
    for row in rows:
        key = (row['seed'], row['preference_diversity'], row['mobility'], row['behaviour'])
        by_seed.setdefault(key, {})
        by_seed[key][row['social_influence']] = row[metric]
    differences = [
        vals[True] - vals[False]
        for vals in by_seed.values()
        if True in vals and False in vals
    ]
    if not differences:
        print(f'Skipped: {title} (no paired data).')
        return
    mean_diff = statistics.fmean(differences)
    ci = 1.96 * statistics.stdev(differences) / math.sqrt(len(differences)) if len(differences) > 1 else 0.0
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.bar(['Social influence effect'], [mean_diff], yerr=[ci], capsize=4, color='#4C78A8')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set(title=title, ylabel=ylabel)
    ax.grid(axis='y', alpha=0.25)
    plt.tight_layout()
    plt.show()
    print(f'Paired differences: n={len(differences)}, mean={mean_diff:.4f}, 95% CI ±{ci:.4f}')

paired_bar_plot(
    social_rows, 'final_segregation_index',
    'Paired effect of social influence on segregation',
    'Mean difference in segregation index (on − off)'
)

**Figure caption.** Paired-seed comparison of the social-influence extension. Final preference standard deviation and total preference updates should be checked alongside segregation.

In [ ]:
def convergence_plot(rows, maximum_runs=3):
    if not rows:
        print('Skipped: convergence plot (no history data).')
        return
    by_run = defaultdict(list)
    for row in rows:
        by_run[row['run_id']].append(row)
    fig, ax = plt.subplots(figsize=(8, 4))
    for run_id, values in list(by_run.items())[:maximum_runs]:
        values.sort(key=lambda row: row['iteration'])
        label = f"{values[0]['preference_diversity']}, {values[0]['behaviour']}, m={values[0]['mobility']}"
        ax.plot([row['iteration'] for row in values],
                [row['segregation_index'] for row in values], label=label)
    ax.set(title='Segregation convergence for selected runs',
           xlabel='Iteration', ylabel='Segregation index')
    ax.legend()
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

convergence_plot(history_rows)

**Figure caption.** Example trajectories show how the segregation index changes during individual simulations. These examples illustrate dynamics and are not substitutes for the repeated-seed summaries.

## 8. Discussion of interactions between variables

In [ ]:
def report_extremes(rows, metric, label):
    if not rows:
        print(f'{label}: awaiting final result CSV.')
        return
    grouped = defaultdict(list)
    for row in rows:
        key = (row['preference_diversity'], row['mobility'], row['behaviour'],
               row['social_influence'])
        grouped[key].append(row[metric])
    means = {key: statistics.fmean(values) for key, values in grouped.items()}
    lowest = min(means, key=means.get)
    highest = max(means, key=means.get)
    print(f'{label} lowest mean: {means[lowest]:.3f} at {lowest}')
    print(f'{label} highest mean: {means[highest]:.3f} at {highest}')
    print(f'Descriptive range between condition means: {means[highest] - means[lowest]:.3f}')

report_extremes(main_rows, 'final_segregation_index', 'Final segregation')
report_extremes(main_rows, 'iterations', 'Stopping time')

Interpret the completed figures by addressing: (1) whether diversity changes outcomes while the mean preference remains fixed; (2) whether mobility has the same effect for each behaviour; (3) whether interaction patterns are consistent across seeds; (4) whether social influence reduces final preference variation; and (5) whether non-stabilised runs change the interpretation of stopping time. Do not claim support for a hypothesis until the final CSV files have been generated and validated.

## 9. Limitations and future work

Important limitations include the finite grid, two-group representation, chosen factor levels, synchronous preference updates, a single vacancy rate and group split, and dependence between repeated observations from paired seeds. Future work could test alternative neighbourhoods, vacancy rates, group proportions, influence strengths, larger grids, and formal factorial models with interaction terms.

## 10. Conclusions

Complete this section only after the final experiment has run. State the strongest reproducible findings, relate them directly to the research question, report important uncertainty or exceptions, and avoid generalising beyond the tested model assumptions.